# 1. 通用递归切割器
- 按字符列表切割
- 按字符数

In [14]:
with open('data/test.txt', encoding='utf-8') as f:
    text_data = f.read()

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 默认分隔符 ["\n\n", "\n", " ", ""]
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                                    chunk_overlap=20,
                                                    # 默认使用字符长度来切割 也可以使用词数来切割
                                                    length_function=len,
                                                    # 是否使用正则表达式来切割
                                                    is_separator_regex=False,
                                                    separators=[
                                                        "\n\n",
                                                        "\n",
                                                        ".",
                                                        "?",
                                                        "!",
                                                        "。",
                                                        "！",
                                                        "？",
                                                        ",",
                                                        "，",
                                                        " "
                                                    ]
                                                    )
recursive_chunks = recursive_splitter.create_documents([text_data])
print(f"分块数量：{len(recursive_chunks)}")

分块数量：154


In [16]:
recursive_chunks[:2]

[Document(metadata={}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman'),
 Document(metadata={}, page_content='. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.')]

# 2. 文本结构切割
- 按章节切割

In [17]:
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>Foo</h1>

        <p>Some intro text about Foo.</p>
        <div>
            <h2>Bar main section</h2>
            <p>Some intro text about Bar.</p>
            <h3>Bar subsection 1</h3>
            <p>Some text about the first subtopic of Bar.</p>
            <h3>Bar subsection 2</h3>
            <p>Some text about the second subtopic of Bar.</p>
        </div>
        <div>
            <h2>Baz</h2>
            <p>Some text about Baz</p>
        </div>
        <br>
        <p>Some concluding text about Foo</p>
    </div>
</body>
</html>
"""

In [20]:
from langchain_text_splitters import HTMLHeaderTextSplitter

label_split = [
    ("h1", '一级章节'),
    ("h2", '二级章节'),
    ("h3", '三级章节'),
]
html_splitter = HTMLHeaderTextSplitter(label_split)
html_chunks = html_splitter.split_text(html_string)
print(f"分块数量：{len(html_chunks)}")

分块数量：10


In [22]:
html_chunks[:]

[Document(metadata={'一级章节': 'Foo'}, page_content='Foo'),
 Document(metadata={'一级章节': 'Foo'}, page_content='Some intro text about Foo.'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section'}, page_content='Bar main section'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section'}, page_content='Some intro text about Bar.'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section', '三级章节': 'Bar subsection 1'}, page_content='Bar subsection 1'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section', '三级章节': 'Bar subsection 1'}, page_content='Some text about the first subtopic of Bar.'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section', '三级章节': 'Bar subsection 2'}, page_content='Bar subsection 2'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar main section', '三级章节': 'Bar subsection 2'}, page_content='Some text about the second subtopic of Bar.'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Baz'}, page_content='Baz'),
 Document(metadata={'一级章节': 

# 3. 多层切割
- 先使用结构切割
- 再使用文本大小切割

In [27]:
label_split_2 = [
    ("h1", '一级章节'),
    ("h2", '二级章节'),
    ("h3", '三级章节'),
    ('h4', '四级章节')
]
html_splitter = HTMLHeaderTextSplitter(label_split_2)

docs_list = html_splitter.split_text_from_url('https://plato.stanford.edu/entries/goedel/')
print(f"分块数量：{len(docs_list)}")

分块数量：75


In [28]:
docs_list[:4]

[Document(metadata={}, page_content='End container NOTE: Script required for drop-down button to work (mirrors).  \nEnd header wrapper End content End footer  \nEnd header  \nEnd navigation End search  \nStanford Encyclopedia of Philosophy  \nMenu  \nBrowse  \nTable of Contents  \nWhat\'s New  \nRandom Entry  \nChronological  \nArchives  \nAbout  \nEditorial Information  \nAbout the SEP  \nEditorial Board  \nHow to Cite the SEP  \nSpecial Characters  \nAdvanced Tools  \nContact  \nSupport SEP  \nSupport the SEP  \nPDFs for SEP Friends  \nMake a Donation  \nSEPIA for Libraries  \nBegin article sidebar End article sidebar NOTE: Article content must have two wrapper divs: id="article" and id="article-content" End article NOTE: article banner is outside of the id="article" div. End article-banner  \nEntry Navigation  \nEntry Contents  \nBibliography  \nAcademic Tools  \nFriends PDF Preview  \nAuthor and Citation Info  \nBack to Top  \nEnd article-content  \nBEGIN ARTICLE HTML #aueditable D

In [29]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=600,
                                                    chunk_overlap=20,
                                                    # 默认使用字符长度来切割 也可以使用词数来切割
                                                    length_function=len,
                                                    # 是否使用正则表达式来切割
                                                    is_separator_regex=False,
                                                    separators=[
                                                        "\n\n",
                                                        "\n",
                                                        ".",
                                                        "?",
                                                        "!",
                                                        "。",
                                                        "！",
                                                        "？",
                                                        ",",
                                                        "，",
                                                        " "
                                                    ]
                                                    )
docs_list2 = recursive_splitter.split_documents(docs_list)
print(f"分块数量：{len(docs_list2)}")

分块数量：280


In [30]:
docs_list2[:4]

[Document(metadata={}, page_content="End container NOTE: Script required for drop-down button to work (mirrors).  \nEnd header wrapper End content End footer  \nEnd header  \nEnd navigation End search  \nStanford Encyclopedia of Philosophy  \nMenu  \nBrowse  \nTable of Contents  \nWhat's New  \nRandom Entry  \nChronological  \nArchives  \nAbout  \nEditorial Information  \nAbout the SEP  \nEditorial Board  \nHow to Cite the SEP  \nSpecial Characters  \nAdvanced Tools  \nContact  \nSupport SEP  \nSupport the SEP  \nPDFs for SEP Friends  \nMake a Donation  \nSEPIA for Libraries"),
 Document(metadata={}, page_content='Begin article sidebar End article sidebar NOTE: Article content must have two wrapper divs: id="article" and id="article-content" End article NOTE: article banner is outside of the id="article" div. End article-banner  \nEntry Navigation  \nEntry Contents  \nBibliography  \nAcademic Tools  \nFriends PDF Preview  \nAuthor and Citation Info  \nBack to Top  \nEnd article-content

# 4. Markdown文档切割

In [31]:
with open('data/Foo.md', encoding='utf-8') as f:
    md_data = f.read()

In [32]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

label_split = [
    ("#", '一级章节'),
    ("##", '二级章节'),
    ("###", '三级章节'),
]
markdown_splitter = MarkdownHeaderTextSplitter(label_split)
markdown_chunks = markdown_splitter.split_text(md_data)
print(f"分块数量：{len(markdown_chunks)}")

分块数量：3


In [33]:
markdown_chunks

[Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar'}, page_content='Hi this is Jim  \nHi this is Joe'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar', '三级章节': 'Boo'}, page_content='Hi this is Lance'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Baz'}, page_content='Hi this is Molly')]

In [35]:
markdown_splitter = MarkdownHeaderTextSplitter(label_split,
                                               # 是否在内容中去掉标题
                                               strip_headers=False
                                               )
markdown_chunks = markdown_splitter.split_text(md_data)
markdown_chunks

[Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar'}, page_content='# Foo  \n## Bar\nHi this is Jim  \nHi this is Joe'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Bar', '三级章节': 'Boo'}, page_content='### Boo\nHi this is Lance'),
 Document(metadata={'一级章节': 'Foo', '二级章节': 'Baz'}, page_content='## Baz\nHi this is Molly')]

# 5. 根据语义切割

In [37]:
import os
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='Qwen/Qwen3-Embedding-8B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

with open('data/test.txt', encoding='utf-8') as f:
    text_data = f.read()

In [40]:
from langchain_experimental.text_splitter import SemanticChunker

# 百分位切割
semantic_chunker = SemanticChunker(embedding, breakpoint_threshold_type='percentile')
chunks = semantic_chunker.create_documents([text_data])
print(f"分块数量：{len(chunks)}")

分块数量：7


In [41]:
chunks

[Document(metadata={}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans. Last year COVID-19 kept us apart. This year we are finally together again. Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. With a duty to one another to the American people to the Constitution. And with an unwavering resolve that freedom will always triumph over tyranny. Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. He met the Ukrainian people. From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. Groups of citizens blocking tanks with t